# 04 - Adapt to Almeria and measure it there

Uses the labelling sheet from notebook 03. The sheet's `split` column assigns every window to train / val / test by
the **2 km block** it lies in, so test areas are geographically separate from training areas.

1. **Before:** the AerialWaste model (notebook 01) on the Almeria test blocks.
2. **Fine-tune** on Almeria train blocks + AerialWaste training images (so it keeps the general knowledge).
   Best epoch and threshold chosen on Almeria validation blocks.
3. **After:** the same Almeria test blocks, scored once. Numbers come with 95 % bootstrap intervals because the
   local test set is small.

In [ ]:
# Setup: GPU, Google Drive (click "Connect to Google Drive" when asked), code
import os, sys, subprocess, json, pathlib
import shutil as _sh
if _sh.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
RUNS = pathlib.Path("/content/drive/MyDrive/waste_runs"); RUNS.mkdir(parents=True, exist_ok=True)
REPO = "/content/waste_detection"
if not os.path.exists(REPO):
    !git clone -q https://github.com/inayatarshad/waste_detection.git {REPO}
else:
    !git -C {REPO} pull -q
!pip install -q -r {REPO}/requirements.txt
sys.path.insert(0, f"{REPO}/src")
import numpy as np, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cpu":
    print("NOTE: no GPU in this session. Testing an existing model works on CPU (slower; "
          "test-time augmentation is switched off). Training needs a GPU.")

In [ ]:
from wastedet import aerialwaste as AW, metrics as M
from wastedet.almeria import almeria_datasets, MixedDataset
from wastedet.train_cls import ClsConfig, load_classifier, predict, train_classifier
from wastedet.scan import read_sheet
SHEETS = sorted((RUNS / "almeria_labels").glob("sheet_*.csv"))
print("sheets:", [s.name for s in SHEETS])
SHEET = SHEETS[0]                      # or merge several areas into one csv first
base, bck = load_classifier(RUNS / "classifier_v1" / "best.pt", DEVICE)
TYPES = bck["types"]
alm = almeria_datasets(SHEET, RUNS / "almeria_labels" / "crops", len(TYPES), size=640)
for k, d in alm.items():
    rs = d.records
    print(f"{k:5s} {len(rs):4d} windows | waste {sum(r.waste for r in rs):3d} | plastic {int(sum(r.types[0] for r in rs))}")
assert min(sum(r.waste for r in alm[k].records) for k in ("train", "val", "test")) >= 5, \
    "too few waste windows in some split - label more (notebook 03) before fine-tuning"

In [ ]:
def almeria_report(model, ds, thr, greenhouse):
    p = predict(model, ds, 32, DEVICE, tta=True, num_workers=2, amp=True)
    y, s = p["y_waste"], p["s_waste"]
    r = {"ap": M.average_precision(y, s), "ap_95ci": M.bootstrap_ci(y, s), **M.binary_at(y, s, thr)}
    g = greenhouse & (y == 0)
    r["false_alarm_rate_greenhouse_windows"] = float((s[g] >= thr).mean()) if g.any() else None
    r["greenhouse_no_waste_windows"] = int(g.sum())
    k = p["mask"][:, 0] > 0
    r["plastic_ap"] = M.average_precision(p["y_types"][k, 0], p["s_types"][k, 0])
    return r, p
rows = {r["file"]: r for r in read_sheet(SHEET)}
gh = lambda ds: np.array([rows[r.file_name].get("greenhouse", "") == "1" for r in ds.records])
before, _ = almeria_report(base, alm["test"], bck["thresholds"]["waste"], gh(alm["test"]))
print("BEFORE (AerialWaste model):", json.dumps(before, indent=1))

In [ ]:
# Fine-tune: Almeria train windows (repeated) + AerialWaste training images
META = RUNS / "aerialwaste_meta"; CACHE = pathlib.Path("/content/aw640")
if not CACHE.exists():
    import tarfile; tarfile.open(RUNS / "aerialwaste_640.tar").extractall("/content")
aw_train = AW.load_records(next(META.glob("*train*.json")), TYPES)
aw_ds = AW.WasteSceneDataset(aw_train, CACHE, 640, train=True)
rep = max(1, len(aw_train) // (4 * max(1, len(alm["train"]))))     # Almeria ~ 20 % of each epoch
mix = MixedDataset([aw_ds, alm["train"]], repeats=[1, rep])
cfg = ClsConfig(types=TYPES, size=640, epochs=6, batch_size=16, lr=5e-5, num_workers=2,
                init_from=str(RUNS / "classifier_v1" / "best.pt"))
OUT = RUNS / "classifier_almeria_v1"
res = train_classifier(cfg, mix, alm["val"], OUT)

In [ ]:
model, ck = load_classifier(OUT / "best.pt", DEVICE)
after_path = OUT / "almeria_test_report.json"
assert not after_path.exists(), "Almeria test blocks already scored"
after, _ = almeria_report(model, alm["test"], ck["thresholds"]["waste"], gh(alm["test"]))
after_path.write_text(json.dumps({"before": before, "after": after}, indent=1, default=float))
print(f"{'':38s}{'before':>10s}{'after':>10s}")
for k in ("ap", "precision", "recall", "f1", "false_alarm_rate", "false_alarm_rate_greenhouse_windows", "plastic_ap"):
    b, a = before.get(k), after.get(k)
    fmt = lambda v: "n/a" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f}"
    print(f"{k:38s}{fmt(b):>10s}{fmt(a):>10s}")
print("95% interval of AP before", np.round(before["ap_95ci"], 3), "after", np.round(after["ap_95ci"], 3))
from wastedet.models import export_classifier_onnx
export_classifier_onnx(model.float().cpu(), OUT / "classifier_almeria.onnx", 640)